# EDA: `raw_airlines`

One notebook per raw Skytrax table (index in `eda.ipynb`). 

**Role of this table:** Dimension table of airlines. `airline_id` is the join key used by every review table.

**Read-only:** the CSV in `data/` is only read. Cleaning below happens on an in-memory copy (`df`); the real cleaning rules are written later in dbt staging.

## Table of Contents

- [0. Setup](#sec-0)
  - [0.1 Imports](#sec-0-1)
  - [0.2 Config](#sec-0-2)
  - [0.3 Plot style](#sec-0-3)
  - [0.4 Helpers](#sec-0-4)
- [1. Look at the Big Picture](#sec-1)
  - [1.1 Frame the Problem](#sec-1-1)
  - [1.2 Questions & Hypotheses](#sec-1-2)
  - [1.3 Performance Measure (for later modelling)](#sec-1-3)
  - [1.4 Check the Assumptions](#sec-1-4)
- [2. Get the Data](#sec-2)
  - [2.1 Data Dictionary](#sec-2-1)
- [3. Data Cleaning (General)](#sec-3)
  - [3.1 Format Column Names](#sec-3-1)
  - [3.2 Format Data Body](#sec-3-2)
  - [3.3 Format Datatypes](#sec-3-3)
  - [3.4 Duplicates](#sec-3-4)
  - [3.5 Domain Sanity Checks](#sec-3-5)
- [4. Data Quality](#sec-4)
  - [4.1 Missing Values](#sec-4-1)
  - [4.2 Outliers](#sec-4-2)
- [5. Attribute Combination](#sec-5)
- [6. EDA](#sec-6)
  - [6.0 Classify Attributes](#sec-6-0)
  - [6.1 Univariate](#sec-6-1)
  - [6.2 Bivariate](#sec-6-2)
  - [6.3 Multivariate](#sec-6-3)
- [7. Summary of Findings](#sec-7)
- [8. Hand-off to Modelling](#sec-8)

# 0. Setup<a id="sec-0"></a>

## 0.1 Imports<a id="sec-0-1"></a>

In [ ]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

## 0.2 Config<a id="sec-0-2"></a>

Everything dataset-specific lives here. Column names are the **snake_case** names produced in section 3.1.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TABLE = "airlines"            # name of the file in data/
DATA_PATH = ROOT / "data" / f"{TABLE}.csv"
TARGET = None                 # dimension table: no target
PROBLEM_TYPE = None           # "regression" or "classification" only when TARGET is set
ID_COLS = ['airline_id']      # identifiers: excluded from analysis
CAT_COLS = None               # force these to categorical; None = auto-detect
CAT_MAX_UNIQUE = 15           # numeric columns with <= this many values are treated as discrete
HIGH_CORR = 0.8               # |r| above this is flagged as multicollinearity
TOP_N_CATEGORIES = 10         # categories shown per bar chart

SAVE_FIGS = False             # True writes every figure to FIG_DIR
FIG_DIR = ROOT / "reports" / "figures"
RANDOM_STATE = 42

## 0.3 Plot style<a id="sec-0-3"></a>

One consistent look: blue for single-series charts, a fixed categorical order for
hue splits (max 3 groups per chart), and a blue ↔ gray ↔ red diverging map for correlations.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MAIN = PALETTE[0]
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#184f95", "#86b6ef", "#f0efec", "#f19a99", "#b52f2e"])

sns.set_theme(
    style="whitegrid",
    palette=PALETTE,
    rc={
        "axes.edgecolor": "#c3c2b7",
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.labelcolor": "#52514e",
        "grid.color": "#ecebe7",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "lines.linewidth": 2,
        "figure.dpi": 100,
    },
)

## 0.4 Helpers<a id="sec-0-4"></a>

In [ ]:
def to_snake_case(df):
    """CamelCase / spaces / dashes -> snake_case column names."""
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.replace(r"(?<=[a-z0-9])(?=[A-Z])", "_", regex=True)
        .str.replace(r"[\s\-/]+", "_", regex=True)
        .str.lower()
    )
    return df


def save_fig(fig, name):
    if SAVE_FIGS and name:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{name}.png", dpi=110, bbox_inches="tight")


def plot_grid(columns, plot_fn, ncols=4, height=3.0, title=None, name=None, shared_legend=False):
    """Draw plot_fn(col, ax) for every column in a grid of subplots.

    shared_legend=True replaces per-panel legends with one legend above the grid.
    """
    columns = list(columns)
    if not columns:
        print("Nothing to plot.")
        return
    nrows = math.ceil(len(columns) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, height * nrows), squeeze=False)
    axes = axes.ravel()
    for ax, col in zip(axes, columns):
        plot_fn(col, ax)
        ax.set_title(col)
    for ax in axes[len(columns):]:
        ax.axis("off")
    fig.tight_layout()
    legend = axes[0].get_legend()
    if shared_legend and legend:
        handles, labels = legend.legend_handles, [t.get_text() for t in legend.get_texts()]
        fig.legend(handles, labels, title=legend.get_title().get_text(), loc="lower center",
                   bbox_to_anchor=(0.5, 1.0), ncol=len(labels), frameon=False)
        for ax in axes:
            if ax.get_legend():
                ax.get_legend().remove()
    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.0 + (0.7 / fig.get_figheight() if shared_legend and legend else 0))
    save_fig(fig, name)
    plt.show()


def missing_report(df):
    """Count and % of nulls per column (only columns with nulls)."""
    report = pd.DataFrame({
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean() * 100,
        "dtype": df.dtypes.astype(str),
    })
    return report[report.n_missing > 0].sort_values("n_missing", ascending=False)


def outlier_report(df, columns, k=1.5):
    """IQR-based outlier counts. Detection only - nothing is removed."""
    rows = []
    for col in columns:
        s = df[col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - k * iqr, q3 + k * iqr
        n_out = int(((s < lo) | (s > hi)).sum())
        rows.append({"column": col, "lower": lo, "upper": hi, "min": s.min(), "max": s.max(),
                     "n_outliers": n_out, "pct_outliers": 100 * n_out / len(s)})
    return pd.DataFrame(rows).set_index("column").sort_values("n_outliers", ascending=False)


def classify_columns(df, target=None, id_cols=(), cat_cols=None, cat_max_unique=15):
    """Split columns into constant / binary / categorical / discrete / continuous."""
    groups = {"constant": [], "binary": [], "categorical": [], "discrete": [], "continuous": []}
    forced_cat = set(cat_cols or [])
    for col in df.columns:
        if col in id_cols or col == target:
            continue
        n = df[col].nunique(dropna=True)
        if n <= 1:
            groups["constant"].append(col)
        elif n == 2:
            groups["binary"].append(col)
        elif col in forced_cat or not pd.api.types.is_numeric_dtype(df[col]):
            groups["categorical"].append(col)
        elif n <= cat_max_unique:
            groups["discrete"].append(col)
        else:
            groups["continuous"].append(col)
    return groups

# 1. Look at the Big Picture<a id="sec-1"></a>

## 1.1 Frame the Problem<a id="sec-1-1"></a>

Answered before touching the data. Business context: the stakeholder is the Head of Customer Experience of a mid-size full-service airline, who decides how to split the yearly budget for improving the customer experience.

* **What is the business objective?**
  Provide a trustworthy list of airlines that every airline review joins to, so scores can be compared between airlines.
* **How will the output be used?**
  Reviews are compared between the airline and its peers; this table is the list of who is who.
* **What does the target look like?** (regression / classification / none)
  Dimension table, no target. It only has to be a reliable list that the review tables join to.
* **What would a useful result be?**
  A yes/no on whether the list can be used as it is, and what has to be fixed first.

## 1.2 Questions & Hypotheses<a id="sec-1-2"></a>

The questions this EDA must answer. Each one maps to a section below. Q-numbers follow the business question list; `D1` is the data-readiness check that comes first.

| # | Question | Answered in |
|---|---|---|
| D1 | Is the data complete and trustworthy (nulls, duplicates, typos)? | 3, 4 |
| Q1 | Is `airline_id` a sound key (unique, no gaps)? | 3.4, 3.5 |
| Q2 | Are names written consistently, and are there leftovers like `Read more`? | 3.2, 3.5 |
| Q3 | Can airlines be grouped into peers with what is in this table? | 7 |

## 1.3 Performance Measure (for later modelling)<a id="sec-1-3"></a>

Pick now so the EDA looks at the right things (e.g. skewed target → consider RMSLE / log target).

* **Regression:** RMSE, MAE, MAPE, RMSLE, adjusted R²
* **Classification:** ROC-AUC, PR-AUC, F1, precision/recall, confusion matrix (check class balance in 6.1)

*Not used here: dimension table, no model.*

## 1.4 Check the Assumptions<a id="sec-1-4"></a>

To confirm with the data owner / stakeholder.

* `airline_id` is the primary key (unique, never null).
* One row = one airline.
* The pseudo-airline `Read more` (a scraping leftover) may be present; it is checked in 3.5.
* The data has no service model (full-service or low-cost), region or peer group; they have to be added from outside.

# 2. Get the Data<a id="sec-2"></a>

In [ ]:
raw = pd.read_csv(DATA_PATH)   # the CSV is only read, never changed
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")

In [ ]:
# Take a look at the top 5 rows
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe(include="all").T

## 2.1 Data Dictionary<a id="sec-2-1"></a>

One row per column: what it means, its type, how many distinct values, and an example. The meanings are inferred from the column names and the data.

In [ ]:
# Meaning of each column (from the column name and the data; correct them if you know better)
MEANINGS = {
    "airline_id": "Numeric identifier of the airline",
    "airline_name": "Display name of the airline",
}

data_dictionary = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "n_unique": raw.nunique(),
    "example": raw.iloc[0],
    "meaning": pd.Series(MEANINGS),
}).reindex(raw.columns)
data_dictionary

# 3. Data Cleaning (General)<a id="sec-3"></a>

Cleaning that is safe to do on the **whole** dataset (no information from the target or test set is used).

## 3.1 Format Column Names<a id="sec-3-1"></a>

In [ ]:
df = to_snake_case(raw)
df.columns.tolist()

In [ ]:
# Manual renames, if any
# df = df.rename(columns={"unnamed:_0": "id"})

## 3.2 Format Data Body<a id="sec-3-2"></a>

Trim whitespace in text columns, then pull structured fields out of free-text columns.

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())
for col in text_cols:
    print(f"{col}: {df[col].nunique()} unique -> {[str(v)[:40] for v in df[col].unique()[:8]]}")

No structured field hides inside the names, so nothing to extract. The cell below only shows how long the names are.

In [ ]:
df["airline_name"].str.len().describe().to_frame().T

## 3.3 Format Datatypes<a id="sec-3-3"></a>

Check every column has the right type: dates → `datetime64`, flags → `int`/`bool`, labels → `category`.

In [ ]:
df.dtypes.value_counts()

In [ ]:
# Both columns already have the right type (integer id, text name): nothing to convert
df.dtypes

## 3.4 Duplicates<a id="sec-3-4"></a>

In [ ]:
analysis_cols = [c for c in df.columns if c not in ID_COLS]
n_exact = df.duplicated().sum()
n_ignoring_id = df.duplicated(subset=analysis_cols).sum()
print(f"Exact duplicate rows:            {n_exact}")
print(f"Duplicates ignoring ID columns:  {n_ignoring_id}")
df[df.duplicated(subset=analysis_cols, keep=False)]

In [ ]:
# Decide: drop or keep? Duplicates ignoring the ID usually mean the same record was loaded twice.
df = df.drop_duplicates(subset=analysis_cols).reset_index(drop=True)
print(f"{len(df):,} rows after de-duplication")

## 3.5 Domain Sanity Checks<a id="sec-3-5"></a>

Values that are *possible* to store but *impossible* in reality (placeholder dates, wrong-column values, leftovers).
These are not statistical outliers; they are data-entry or scraping errors. Wrong dates are set to missing here, like the staging model will do.

In [ ]:
# Names that are too short to be a real airline
display(df[df["airline_name"].str.len() <= 1])

# The same name with a different case = the same airline twice
name_counts = df.assign(name_key=df["airline_name"].str.lower()).groupby("name_key").size()
print("Names repeated after lower-casing:", int((name_counts > 1).sum()))

# The scraping leftover "Read more" is not an airline
display(df[df["airline_name"].str.lower() == "read more"])

# Ids should run 1..N with no hole (0 = no gap)
print("Id gaps:", int(df["airline_id"].max() - df["airline_id"].min() + 1 - df["airline_id"].nunique()))


# 4. Data Quality<a id="sec-4"></a>

## 4.1 Missing Values<a id="sec-4-1"></a>

**Types of nulls** (decides how to treat them later):
* **MCAR** - missing completely at random (lost in ETL). Drop or impute.
* **MAR** - missingness explained by another observed column. Impute using that column.
* **MNAR** - missingness depends on the missing value itself (e.g. high earners hiding income). Add a "was missing" flag; model carefully.
* **By design** - field not applicable (e.g. no cancellation date for active customers). Fill with a sentinel.

**Options:** drop rows · drop column · impute (median / mode / model-based) · add an `is_missing` indicator.
*Only decide here; fit imputers on the training set after the split.*

In [ ]:
missing = missing_report(df)
print(f"{df.isna().sum().sum()} missing values in {len(missing)} columns; "
      f"{df.isna().any(axis=1).sum()} rows ({df.isna().any(axis=1).mean():.1%}) affected")
missing

In [ ]:
if len(missing):
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(missing) + 1))
    ax.barh(missing.index, missing["pct_missing"], color=MAIN, height=0.6)
    ax.invert_yaxis()
    ax.set_xlabel("% missing")
    ax.set_title("Missing values by column")
    for i, v in enumerate(missing["pct_missing"]):
        ax.text(v, i, f" {v:.1f}%", va="center", color="#52514e", fontsize=9)
    save_fig(fig, "missing_values")
    plt.show()

In [ ]:
# Do nulls co-occur? (same rows missing several fields -> one upstream cause)
if len(missing) > 1:
    display(df[missing.index].isna().corr().round(2))

In [ ]:
# Rows with a missing TARGET cannot be used for supervised training
if TARGET and df[TARGET].isna().any():
    print(f"{df[TARGET].isna().sum()} rows have no {TARGET} -> drop before modelling")

## 4.2 Outliers<a id="sec-4-2"></a>

**Detect, don't delete.** Flag values outside `Q1 - 1.5·IQR` / `Q3 + 1.5·IQR`, then decide per column:
real but rare (keep, maybe log-transform) vs. error (fix in 3.5).

In [ ]:
num_cols = [c for c in df.select_dtypes("number").columns
            if c not in ID_COLS and df[c].nunique() > 2]
if num_cols:
    outliers = outlier_report(df, num_cols)
    display(outliers[outliers.n_outliers > 0])
else:
    print("No numeric columns besides the id: nothing to check.")

In [ ]:
plot_grid(num_cols, lambda c, ax: sns.boxplot(x=df[c], ax=ax, color=MAIN, fliersize=3, linewidth=1),
          title="Box plots (numeric columns)", name="boxplots")

# 5. Attribute Combination<a id="sec-5"></a>

Create features that may explain the data better than the raw columns (ratios, rates, flags, date parts).
They go through the same EDA as everything else below.

names get a length column.

In [ ]:
# Dimension tables have nothing to combine; name length helps spot odd names
df["name_length"] = df["airline_name"].str.len()


# 6. EDA<a id="sec-6"></a>

## 6.0 Classify Attributes<a id="sec-6-0"></a>

Every plot below is driven by these groups. Override with `CAT_COLS` in the config if a group looks wrong.

In [ ]:
groups = classify_columns(df, TARGET, ID_COLS, CAT_COLS, CAT_MAX_UNIQUE)
for name, cols in groups.items():
    print(f"{name:<12} ({len(cols):>2}): {cols}")

const_attribs = groups["constant"]
bin_attribs = groups["binary"]
cat_attribs = groups["categorical"]
disc_attribs = groups["discrete"]
cont_attribs = groups["continuous"]
num_attribs = disc_attribs + cont_attribs

In [ ]:
# Constant columns carry no information
if const_attribs:
    print("Dropping constant columns:", const_attribs)
    df = df.drop(columns=const_attribs)

# High-cardinality text columns (e.g. free text, raw model names) swamp plots
high_card = [c for c in cat_attribs if df[c].nunique() > 50]
print("High-cardinality categoricals (summarised, not plotted):", high_card)
cat_plot_attribs = [c for c in cat_attribs if c not in high_card]

## 6.1 Univariate<a id="sec-6-1"></a>

One column at a time: shape, spread, skew, rare levels.

### 6.1.1 Target

Dimension table, `TARGET = None`: the next cell prints nothing.

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    y = df[TARGET].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(y, kde=True, ax=axes[0], color=MAIN)
    axes[0].set_title(f"{TARGET} (skew = {y.skew():.2f})")
    sns.histplot(np.log1p(y), kde=True, ax=axes[1], color=MAIN)
    axes[1].set_title(f"log1p({TARGET}) (skew = {np.log1p(y).skew():.2f})")
    fig.tight_layout()
    save_fig(fig, "target_distribution")
    plt.show()
    display(y.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame())
elif TARGET:
    counts = df[TARGET].value_counts()
    display(pd.DataFrame({"count": counts, "pct": counts / counts.sum() * 100}))
    fig, ax = plt.subplots(figsize=(6, 3))
    counts.plot.bar(ax=ax, color=MAIN)
    ax.set_title(f"Class balance: {TARGET}")
    save_fig(fig, "target_distribution")
    plt.show()

### 6.1.2 Numeric

In [ ]:
skew = df[num_attribs].skew().sort_values(key=abs, ascending=False).to_frame("skew")
skew["shape"] = pd.cut(skew["skew"].abs(), [-np.inf, 0.5, 1, np.inf], labels=["symmetric", "moderate", "highly skewed"])
skew

In [ ]:
plot_grid(cont_attribs, lambda c, ax: sns.histplot(df[c], kde=True, ax=ax, color=MAIN),
          title="Continuous features", name="hist_continuous")

In [ ]:
plot_grid(disc_attribs, lambda c, ax: sns.countplot(x=df[c], ax=ax, color=MAIN),
          title="Discrete features", name="count_discrete")

### 6.1.3 Binary

In [ ]:
if bin_attribs:
    binary_share = (df[bin_attribs]
                    .apply(lambda s: s.value_counts(normalize=True).iloc[0] * 100)
                    .sort_values(ascending=False)
                    .to_frame("pct_majority_value"))
    binary_share["near_constant"] = binary_share["pct_majority_value"] > 95
    display(binary_share)
else:
    print("No binary columns.")

In [ ]:
plot_grid(bin_attribs, lambda c, ax: sns.countplot(x=df[c], ax=ax, color=MAIN),
          ncols=6, height=2.4, title="Binary features", name="count_binary")

### 6.1.4 Categorical

In [ ]:
for col in high_card:
    print(f"{col}: {df[col].nunique()} levels; top 5:")
    display(df[col].value_counts().head(5).to_frame())

In [ ]:
def plot_top_categories(col, ax):
    counts = df[col].value_counts()
    top = counts.head(TOP_N_CATEGORIES)
    sns.barplot(x=top.values, y=top.index.astype(str), ax=ax, color=MAIN, orient="h")
    ax.set_xlabel("count")
    ax.set_ylabel("")

plot_grid(cat_plot_attribs, plot_top_categories, ncols=3, height=3.5,
          title="Categorical features (top levels)", name="count_categorical")

In [ ]:
# Rare levels (< 1% of rows) - candidates for grouping into "other"
rare = {c: df[c].value_counts(normalize=True).loc[lambda s: s < 0.01].index.tolist() for c in cat_plot_attribs}
{c: v for c, v in rare.items() if v}

## 6.2 Bivariate<a id="sec-6-2"></a>

Dimension table, `TARGET = None`: the cells below print nothing.

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    target_corr = (df[num_attribs + bin_attribs + [TARGET]]
                   .corr(method="spearman")[TARGET]
                   .drop(TARGET)
                   .sort_values(key=abs, ascending=False))
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(target_corr) + 1))
    colors = np.where(target_corr > 0, PALETTE[7], PALETTE[0])
    ax.barh(target_corr.index, target_corr.values, color=colors, height=0.7)
    ax.axvline(0, color="#52514e", linewidth=1)
    ax.invert_yaxis()
    ax.set_xlabel(f"Spearman correlation with {TARGET}")
    ax.set_title(f"Features ranked by correlation with {TARGET}")
    save_fig(fig, "target_correlation")
    plt.show()

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    top_num = [c for c in target_corr.index if c in cont_attribs][:8]
    plot_grid(top_num,
              lambda c, ax: sns.regplot(data=df, x=c, y=TARGET, ax=ax, lowess=True,
                                        scatter_kws={"s": 8, "alpha": 0.35, "color": MAIN},
                                        line_kws={"color": PALETTE[1]}),
              title=f"Top continuous features vs {TARGET} (LOWESS trend)", name="scatter_vs_target")
elif TARGET:
    plot_grid(cont_attribs,
              lambda c, ax: sns.kdeplot(data=df, x=c, hue=TARGET, common_norm=False, fill=True,
                                        alpha=0.25, linewidth=2, ax=ax),
              title=f"Continuous features by {TARGET} class", name="dist_by_target", shared_legend=True)

In [ ]:
def plot_cat_vs_target(col, ax):
    order = df[col].value_counts().head(TOP_N_CATEGORIES).index
    if pd.api.types.is_numeric_dtype(df[col]):
        order = order.sort_values()
    if PROBLEM_TYPE == "regression":
        sns.boxplot(data=df[df[col].isin(order)], x=TARGET, y=col, order=order,
                    ax=ax, color=MAIN, fliersize=2, linewidth=1, orient="h")
    else:
        rate = pd.crosstab(df[col], df[TARGET], normalize="index").loc[order]
        rate.plot.barh(stacked=True, ax=ax, width=0.7, edgecolor="white", linewidth=1)
        ax.set_xlabel("share of rows")
    ax.set_ylabel("")

if TARGET:
    by_level = cat_plot_attribs + disc_attribs + (bin_attribs if PROBLEM_TYPE == "classification" else [])
    plot_grid(by_level, plot_cat_vs_target, ncols=3, height=3.5,
              title=f"Categorical / discrete features vs {TARGET}", name="cat_vs_target", shared_legend=True)

In [ ]:
# Binary flags: difference in median target between the two groups (a = lower value, b = higher)
if TARGET and PROBLEM_TYPE == "regression":
    rows = {}
    for col in bin_attribs:
        a, b = sorted(df[col].dropna().unique())
        y_a, y_b = df.loc[df[col] == a, TARGET].dropna(), df.loc[df[col] == b, TARGET].dropna()
        rows[col] = {"a": a, "b": b, "median_when_a": y_a.median(), "median_when_b": y_b.median(),
                     "share_of_b": (df[col] == b).mean(), "p_value": stats.mannwhitneyu(y_a, y_b).pvalue}
    lift = pd.DataFrame(rows).T
    lift["diff"] = (lift["median_when_b"] - lift["median_when_a"]).astype(float)
    display(lift.sort_values("diff", key=abs, ascending=False))

## 6.3 Multivariate<a id="sec-6-3"></a>

Features against **each other**: redundancy, leakage, interactions.

In [ ]:
corr_cols = num_attribs
if len(corr_cols) >= 2:
    corr = df[corr_cols].corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(0.6 * len(corr) + 3, 0.5 * len(corr) + 2))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=DIVERGING, vmin=-1, vmax=1, center=0,
                linewidths=2, linecolor="white", annot_kws={"size": 8}, cbar_kws={"shrink": 0.6}, ax=ax)
    ax.set_title("Spearman correlation (numeric features)")
    save_fig(fig, "correlation_heatmap")
    plt.show()
else:
    print("Fewer than two numeric columns: no correlation to show.")

In [ ]:
# Highly correlated pairs -> keep one of each pair (or combine) before modelling
if len(corr_cols) >= 2:
    pairs = (corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
             .stack()
             .rename("r")
             .reset_index()
             .rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
    display(pairs[pairs.r.abs() >= HIGH_CORR].sort_values("r", key=abs, ascending=False))

In [ ]:
# Leakage check: ID columns or row order should NOT predict the target
if TARGET and PROBLEM_TYPE == "regression":
    for col in ID_COLS:
        if col in df.columns:
            r = df[[col, TARGET]].corr(method="spearman").iloc[0, 1]
            print(f"corr({col}, {TARGET}) = {r:.2f}" + ("  <-- rows are ordered; shuffle before splitting" if abs(r) > 0.3 else ""))

### 6.3.1 Segments

Not applicable: this dimension table has no segment column and no ratings.

In [ ]:
# Pairplot of the strongest features (slow on wide data - keep it small)
if TARGET and PROBLEM_TYPE == "regression":
    pp_cols = [c for c in target_corr.index if c in cont_attribs][:4] + [TARGET]
    g = sns.pairplot(df[pp_cols].dropna(), corner=True, plot_kws={"s": 8, "alpha": 0.35, "color": MAIN},
                     diag_kws={"color": MAIN})
    g.figure.suptitle("Pairplot: strongest continuous features", y=1.02)
    save_fig(g.figure, "pairplot")
    plt.show()

# 7. Summary of Findings<a id="sec-7"></a>

The deliverable: one answer per question from 1.2, then the actions before staging. Drafted from the data, to be reviewed.
*Numbers come from running this notebook on the CSV. Counts printed from 3.5 onward are taken after the duplicates dropped in 3.4, so they can be slightly lower than counts on the raw file.*

**Answers to the questions in 1.2**
| # | Answer |
|---|---|
| D1 | Clean: 595 rows, no nulls, no duplicates (even ignoring case), no stray spaces. |
| Q1 | Yes: `airline_id` is unique, never null, and runs 1 to 595 with no gap. |
| Q2 | Names are consistent, but one row is a scraping leftover (`Read more`). |
| Q3 | No: the table has only an id and a name. Service model, region and peer group have to be added from outside (a seed). |

**Data quality**
* 595 rows x 2 columns, 0 missing values, 0 duplicate names after lower-casing, 0 id gaps.
* One row is not an airline: `Read more`.

**Target**
* Dimension table: no target.

**Key drivers**
* Not applicable.

**Redundancy / leakage**
* Only two columns, nothing to correlate.

**Actions before staging**
| Action | Column(s) | Why |
|---|---|---|
| Remove the row `Read more` | `airline_name` | not a real airline |
| Treat `airline_id` as primary key (`unique`, `not_null` tests) | `airline_id` | unique and complete |
| Add service model, region and peer group (seed) | new columns | needed to compare with peers |


# 8. Hand-off to Modelling<a id="sec-8"></a>

Next steps (the template's modelling checklist does not apply: there is no model):

1. **Write the staging model** `stg__airlines` with the cleaning actions from section 7.
2. **Add dbt tests** (`unique`, `not_null`, `relationships`, `accepted_values`) for every rule that can be tested.
3. **Run `dbt run` and `dbt test`**, then compare row counts and null counts with the numbers in this notebook.
4. **Build the star schema** (facts and dimensions) on top of the staging models.